# Blood Cell Classifier — Exploratory Data Analysis

This notebook profiles the two available datasets and validates the data-leakage concern described in `PLAN.md`.

**Sections:**
1. Class distribution & image statistics
2. Sample visualizations per class
3. SourceID analysis & leakage quantification
4. Dataset-master (raw smear) overview

In [ ]:
import os, sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA2 = PROJECT_ROOT / "dataset2-master" / "dataset2-master" / "images"
DATA1 = PROJECT_ROOT / "dataset-master" / "dataset-master"

CLASSES = ["EOSINOPHIL", "LYMPHOCYTE", "MONOCYTE", "NEUTROPHIL"]
sns.set_style("whitegrid")
print("Data root:", DATA2)
print("Splits:", os.listdir(DATA2))

## 1. Class Distribution

In [ ]:
# Count images per class per split
rows = []
for split in ["TRAIN", "TEST", "TEST_SIMPLE"]:
    for cls in CLASSES:
        cls_dir = DATA2 / split / cls
        if cls_dir.is_dir():
            count = len([f for f in os.listdir(cls_dir) if f.endswith(('.jpeg', '.jpg', '.png'))])
            rows.append({"Split": split, "Class": cls, "Count": count})

df_counts = pd.DataFrame(rows)
pivot = df_counts.pivot(index="Class", columns="Split", values="Count")
pivot = pivot[["TRAIN", "TEST", "TEST_SIMPLE"]]
print(pivot)
print(f"\nTotals: TRAIN={pivot['TRAIN'].sum()}, TEST={pivot['TEST'].sum()}, TEST_SIMPLE={pivot['TEST_SIMPLE'].sum()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, split in zip(axes, ["TRAIN", "TEST"]):
    sub = df_counts[df_counts["Split"] == split]
    sns.barplot(data=sub, x="Class", y="Count", ax=ax, palette="Set2")
    ax.set_title(f"{split} — Class Distribution")
    ax.set_ylim(0, sub["Count"].max() * 1.15)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{int(bar.get_height())}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Image Size Statistics & Sample Visualizations

In [ ]:
# Check image dimensions
widths, heights = [], []
sample_paths = {}
for cls in CLASSES:
    cls_dir = DATA2 / "TRAIN" / cls
    files = sorted([f for f in os.listdir(cls_dir) if f.endswith('.jpeg')])
    sample_paths[cls] = [cls_dir / f for f in files[:4]]
    for f in files[:200]:  # sample 200 per class
        img = Image.open(cls_dir / f)
        widths.append(img.width)
        heights.append(img.height)

print(f"Image dimensions (sample of {len(widths)} images):")
print(f"  Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"  Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
print(f"  Unique sizes: {set(zip(widths, heights))}")

In [ ]:
# Visualize 4 samples per class
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for i, cls in enumerate(CLASSES):
    for j, path in enumerate(sample_paths[cls]):
        img = Image.open(path)
        axes[i, j].imshow(img)
        axes[i, j].set_title(f"{cls}\n{path.name}", fontsize=8)
        axes[i, j].axis("off")

plt.suptitle("Sample Images per Class (dataset2 TRAIN)", fontsize=14)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "sample_images.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. SourceID Analysis & Leakage Quantification

Filenames follow the pattern `_<sourceID>_<augIndex>.jpeg`. Multiple augmented crops of the same source cell share a sourceID. If the same sourceID appears in both TRAIN and TEST, the model can "memorize" individual cells rather than learning class-level morphology.

In [ ]:
# Extract sourceIDs and analyze leakage
def get_source_ids(split):
    ids = {}
    for cls in CLASSES:
        cls_dir = DATA2 / split / cls
        for f in os.listdir(cls_dir):
            if f.endswith('.jpeg'):
                sid = f.split('_')[1]
                ids.setdefault(sid, []).append((cls, f))
    return ids

train_ids = get_source_ids("TRAIN")
test_ids = get_source_ids("TEST")

overlap = set(train_ids.keys()) & set(test_ids.keys())
only_train = set(train_ids.keys()) - set(test_ids.keys())
only_test = set(test_ids.keys()) - set(train_ids.keys())

print(f"Unique sourceIDs — TRAIN: {len(train_ids)}, TEST: {len(test_ids)}")
print(f"Overlap:    {len(overlap)} IDs ({len(overlap)/len(test_ids)*100:.0f}% of TEST IDs leak into TRAIN)")
print(f"Train-only: {len(only_train)}")
print(f"Test-only:  {len(only_test)}")

# Count how many TEST images are from leaked sourceIDs
leaked_test_imgs = sum(len(test_ids[sid]) for sid in overlap)
total_test_imgs = sum(len(v) for v in test_ids.values())
print(f"\nLeaked test images: {leaked_test_imgs}/{total_test_imgs} ({leaked_test_imgs/total_test_imgs*100:.1f}%)")

# Augmentation expansion factor
augs_per_source = [len(v) for v in train_ids.values()]
print(f"\nAugmentations per sourceID — min: {min(augs_per_source)}, max: {max(augs_per_source)}, "
      f"mean: {np.mean(augs_per_source):.1f}, median: {np.median(augs_per_source):.0f}")

In [ ]:
# Visualize: augmentations of the same sourceID
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
# Pick a sourceID that has many augmentations
sid = sorted(train_ids.keys(), key=lambda s: len(train_ids[s]), reverse=True)[0]
items = train_ids[sid][:10]
for i, (cls, fname) in enumerate(items):
    ax = axes[i // 5, i % 5]
    img = Image.open(DATA2 / "TRAIN" / cls / fname)
    ax.imshow(img)
    ax.set_title(f"sourceID={sid}\n{cls} / {fname}", fontsize=7)
    ax.axis("off")
plt.suptitle(f"Augmentations sharing sourceID={sid} (same cell, different augmentations)", fontsize=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "sourceID_augmentations.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Dataset-Master Overview (Raw Blood Smears)

The raw dataset contains full-frame 640×480 blood smear images with Pascal VOC bounding-box annotations.

In [ ]:
# Dataset-master: labels.csv analysis
labels_csv = DATA1 / "labels.csv"
df_labels = pd.read_csv(labels_csv)
print(f"Raw dataset: {len(df_labels)} rows")
print(f"\nColumns: {list(df_labels.columns)}")
print(f"\nClass distribution:")
print(df_labels["Category"].value_counts())

# Show sample raw smear images
jpeg_dir = DATA1 / "JPEGImages"
raw_files = sorted(os.listdir(jpeg_dir))[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, fname in zip(axes, raw_files):
    img = Image.open(jpeg_dir / fname)
    ax.imshow(img)
    ax.set_title(f"{fname}\n{img.size}", fontsize=8)
    ax.axis("off")
plt.suptitle("Sample Raw Blood Smear Images (dataset-master)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Grouped Split Verification

Verify that our `GroupShuffleSplit` in `src/data.py` produces zero leakage.

In [ ]:
from src.data import get_grouped_split_loaders

# This will print split sizes and assert no group leakage
train_l, val_l, test_l = get_grouped_split_loaders(batch_size=32, num_workers=0)
print(f"\nTrain samples: {len(train_l.dataset)}")
print(f"Val samples:   {len(val_l.dataset)}")
print(f"Test samples:  {len(test_l.dataset)}")
print("\n✓ Group-aware split created with zero leakage.")